# 02 – Preprocess SD7K
Resize SD7K pairs to **512 × 512 JPEG** and create train / val / test splits.

**How to use:**
1. On **Kaggle**: attach your raw SD7K dataset, then *Run All*.
2. On **Colab**: mount Drive first (run notebook 01), then *Run All*.
3. Locally: set `data/sd7k_raw/` → `train/input`, `train/target`, `test/input`, `test/target` and run.

Output goes to `sd7k_512/` with `train / val / test` splits, all images as `.jpg`.

In [ ]:
# ── Cell 1: Imports, helpers, and path detection ────────────────────────────
import os, re, shutil, random
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm
from concurrent.futures import ProcessPoolExecutor

# ── Config ──────────────────────────────────────────────────────────────────
IMAGE_SIZE = 512
JPEG_Q     = 95
VAL_FRAC   = 0.05
SEED       = 42

IMG_EXTS = {'.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff'}

# Folder-name aliases used by different SD7K download sources
INPUT_ALIASES  = {'input','inputs','shadow','shadows','shadow_img','shadow_images'}
TARGET_ALIASES = {'target','targets','gt','clean','shadow_free','shadow-free',
                  'shadowfree','ground_truth','groundtruth','noshadow'}

# ── Helper functions ─────────────────────────────────────────────────────────
def glob_images(d: Path):
    """List all image files in a directory; returns [] if directory missing."""
    if not d or not d.exists() or not d.is_dir():
        return []
    return sorted(p for p in d.iterdir() if p.suffix.lower() in IMG_EXTS)

def resolve_subdir(split_dir: Path, is_input: bool) -> Path:
    """Find the input or target subdirectory inside a split folder."""
    if not split_dir.exists():
        return split_dir / ('input' if is_input else 'target')
    aliases = INPUT_ALIASES if is_input else TARGET_ALIASES
    # Exact / alias match
    for sub in split_dir.iterdir():
        if sub.is_dir() and sub.name.lower() in aliases:
            return sub
    return split_dir / ('input' if is_input else 'target')

def get_split_dir(root: Path, split: str, is_input: bool) -> Path:
    """Resolve root/<split>/<input|target> case-insensitively with alias support."""
    if not root.exists():
        return root / split / ('input' if is_input else 'target')
    for s in root.iterdir():
        if s.is_dir() and s.name.lower() == split.lower():
            return resolve_subdir(s, is_input)
    return root / split / ('input' if is_input else 'target')

def build_lookup(target_dir: Path) -> dict:
    """Build a stem → Path dict for the target directory."""
    lu = {}
    if not target_dir or not target_dir.exists():
        return lu
    for p in target_dir.iterdir():
        if p.is_file() and p.suffix.lower() in IMG_EXTS:
            lu[p.name]        = p   # exact filename
            lu[p.stem]        = p   # stem (no extension)
            lu[p.stem.lower()] = p  # lowercase stem
    return lu

def find_target(inp: Path, lu: dict):
    """Match an input image to its target using flexible name matching."""
    for key in (inp.name, inp.stem, inp.stem.lower(),
                inp.stem.lower().replace('_shadow','').replace('_input','')):
        if key in lu:
            return lu[key]
    return None

def find_sd7k_root() -> Path:
    """Walk /kaggle/input (or data/) to find the folder that has train/<input_alias>/."""
    for base in [Path('/kaggle/input'), Path('data'), Path('.')]:
        if not base.exists():
            continue
        for c in [base] + [p for p in base.rglob('*') if p.is_dir()]:
            try:
                for child in c.iterdir():
                    if child.is_dir() and child.name.lower() in ('train','training'):
                        for sub in child.iterdir():
                            if sub.is_dir() and sub.name.lower() in INPUT_ALIASES:
                                if any(True for img in sub.iterdir()
                                       if img.suffix.lower() in IMG_EXTS):
                                    return c
            except Exception:
                pass
    return None

# ── Path setup ───────────────────────────────────────────────────────────────
KAGGLE = os.path.exists('/kaggle')
COLAB  = 'google.colab' in __import__('sys').modules

if KAGGLE:
    root = find_sd7k_root()
    SD7K_RAW = root if root else Path('/kaggle/input/sd7k-raw')
    OUT_DIR  = Path('/kaggle/working/sd7k_512')
    if root:
        print(f'✅ Auto-detected SD7K at: {SD7K_RAW}')
    else:
        print(f'⚠️  Not found — defaulting to {SD7K_RAW}')
        print('Paths under /kaggle/input:')
        for p in sorted(Path('/kaggle/input').rglob('*'))[:30]: print(f'  {p}')
elif COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    SD7K_RAW = Path('/content/drive/MyDrive/DocRescue/data/sd7k_raw')
    OUT_DIR  = Path('/content/drive/MyDrive/DocRescue/data/sd7k_512')
    print(f'Colab: SD7K raw at {SD7K_RAW}')
else:
    SD7K_RAW = Path('data/sd7k_raw')
    OUT_DIR  = Path('data/sd7k_512')

print(f'SD7K raw : {SD7K_RAW}')
print(f'Output   : {OUT_DIR}')

# Quick folder preview
for split in ('train','test'):
    for is_inp in (True, False):
        d = get_split_dir(SD7K_RAW, split, is_inp)
        label = 'input' if is_inp else 'target'
        n = len(glob_images(d))
        status = '✅' if n > 0 else '❌'
        print(f'  {status} {split}/{label}: {n} images  →  {d}')

In [ ]:
# ── Cell 2: Pair, split, resize — all-in-one ──────────────────────────────
# This cell is fully self-contained; re-runnable without running any other cell first.
import re

def doc_id(filename: str) -> str:
    """Extract a stable document-level ID to prevent train/val leakage."""
    stem = Path(filename).stem
    # Handle 'IMG_0001' style names
    stem = re.sub(r'^IMG_', '', stem, flags=re.IGNORECASE)
    # Take prefix before first underscore (e.g. '0001' from '0001_003')
    m = re.match(r'^([^_]+)', stem)
    return m.group(1) if m else stem

# ---------- Build paired list for TRAIN ----------
tr_in  = get_split_dir(SD7K_RAW, 'train', is_input=True)
tr_tgt = get_split_dir(SD7K_RAW, 'train', is_input=False)
tgt_lu = build_lookup(tr_tgt)

paired, skipped = [], []
for inp in glob_images(tr_in):
    tgt = find_target(inp, tgt_lu)
    if tgt:
        paired.append((inp, tgt))
    else:
        skipped.append(inp.name)

if not paired:
    raise FileNotFoundError(
        f"Could not match any input → target pair.\n"
        f"  Input dir  : {tr_in}  ({len(glob_images(tr_in))} files)\n"
        f"  Target dir : {tr_tgt} ({len(glob_images(tr_tgt))} files)\n"
        f"  Sample inputs : {[p.name for p in glob_images(tr_in)[:5]]}\n"
        f"  Sample targets: {[p.name for p in glob_images(tr_tgt)[:5]]}"
    )

print(f'✅ Matched {len(paired)} training pairs  (skipped {len(skipped)} unmatched inputs)')
if skipped: print(f'  Skipped examples: {skipped[:5]}')

# ---------- Train / val split by document ID ----------
random.seed(SEED)
all_doc_ids = sorted({doc_id(inp.name) for inp, _ in paired})
n_val = max(1, int(len(all_doc_ids) * VAL_FRAC))
val_docs   = set(random.sample(all_doc_ids, n_val))
train_docs = set(all_doc_ids) - val_docs

train_pairs = [(i, t) for i, t in paired if doc_id(i.name) in train_docs]
val_pairs   = [(i, t) for i, t in paired if doc_id(i.name) in val_docs]
print(f'Split → train: {len(train_pairs)} pairs, val: {len(val_pairs)} pairs')

# ---------- Build paired list for TEST ----------
te_in  = get_split_dir(SD7K_RAW, 'test', is_input=True)
te_tgt = get_split_dir(SD7K_RAW, 'test', is_input=False)
te_lu  = build_lookup(te_tgt)
test_pairs = []
for inp in glob_images(te_in):
    tgt = find_target(inp, te_lu)
    if tgt:
        test_pairs.append((inp, tgt))
print(f'Test  → {len(test_pairs)} pairs')

# ---------- Build task list ----------
tasks = []
for split_label, pairs in [('train', train_pairs), ('val', val_pairs), ('test', test_pairs)]:
    for inp, tgt in pairs:
        stem = inp.stem + '.jpg'
        tasks.append((
            str(inp),
            str(tgt),
            str(OUT_DIR / split_label / 'input'  / stem),
            str(OUT_DIR / split_label / 'target' / stem),
        ))

print(f'Total tasks: {len(tasks)} — starting resize …')

# ---------- Worker (runs in subprocess — must NOT use any closures / globals) ----------
def _resize_pair(args):
    from pathlib import Path
    from PIL import Image
    src_i, src_t, dst_i, dst_t = args
    SIZE, Q = 512, 95
    for src, dst in [(Path(src_i), Path(dst_i)), (Path(src_t), Path(dst_t))]:
        Path(dst).parent.mkdir(parents=True, exist_ok=True)
        if Path(dst).exists():
            continue
        with Image.open(src).convert('RGB') as img:
            img.resize((SIZE, SIZE), Image.LANCZOS).save(dst, 'JPEG', quality=Q)

# ---------- Run parallel resize ----------
with ProcessPoolExecutor(max_workers=4) as ex:
    list(tqdm(ex.map(_resize_pair, tasks), total=len(tasks), desc='Resizing'))

print('\n✅ Done! Output sizes:')
for split_label in ('train', 'val', 'test'):
    n = len(list((OUT_DIR / split_label / 'input').glob('*.jpg')))
    print(f'  {split_label}: {n} pairs')

In [ ]:
# ── Cell 3: Sanity-check visualisation ────────────────────────────────────
import matplotlib.pyplot as plt
from PIL import Image

sample = list((OUT_DIR / 'train' / 'input').glob('*.jpg'))[:3]
if not sample:
    print('No output images found yet — run Cell 2 first.')
else:
    fig, axes = plt.subplots(2, len(sample), figsize=(5 * len(sample), 7))
    if len(sample) == 1:
        axes = [[axes[0]], [axes[1]]]
    for col, sp in enumerate(sample):
        s = Image.open(sp)
        c = Image.open(OUT_DIR / 'train' / 'target' / sp.name)
        axes[0][col].imshow(s); axes[0][col].set_title(sp.name[:14], fontsize=7); axes[0][col].axis('off')
        axes[1][col].imshow(c); axes[1][col].axis('off')
    axes[0][0].set_ylabel('Input (shadow)', fontsize=9)
    axes[1][0].set_ylabel('Target (clean)', fontsize=9)
    plt.suptitle(f'Preprocessed SD7K pairs ({IMAGE_SIZE}×{IMAGE_SIZE})')
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Cell 4 (Optional): Zip output for Kaggle Dataset creation ─────────────
if KAGGLE:
    archive = '/kaggle/working/sd7k_512'
    print(f'Archiving {OUT_DIR} → {archive}.zip …')
    shutil.make_archive(archive, 'zip', str(OUT_DIR))
    print(f'Saved: {archive}.zip')
else:
    print('Skipping zip (not on Kaggle).')